# NB01: prepare baselines

Transcribes the public-domain and Earnings speeches with Whisper (word timings), then cuts 45 to 90 s passages at sentence boundaries, screens them and writes `candidates/` on Drive. Both stages resume where they stopped.

Runtime: T4 GPU.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/oratio")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/Jonathan-Jerrick/oratio.git", str(REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{REPO}/packages/speechgrade",
        "-e",
        f"{REPO}/dataset_factory[colab]",
    ],
    capture_output=True,
    text=True,
)
print(install.stdout[-2000:], install.stderr[-4000:])
install.check_returncode()
for src in (REPO / "packages/speechgrade/src", REPO / "dataset_factory/src"):
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))
print(
    subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout
)

In [ ]:
from oratio_data.stages import nb01

ROOT = Path("/content/drive/MyDrive/Oratio")
SCRATCH = Path("/content/scratch")
SCRATCH.mkdir(exist_ok=True)
PER_SOURCE = None  # set to a small number, e.g. 2, for a smoke run

In [ ]:
transcribed = nb01.transcribe(ROOT, SCRATCH, per_source=PER_SOURCE)
print(transcribed)

In [ ]:
cut = nb01.cut(ROOT)
nb01.write_manifest(ROOT, "NB01", {"per_source": PER_SOURCE, "transcribe": transcribed, "cut": cut})
print(cut)

In [ ]:
import pandas as pd

candidates = pd.read_csv(ROOT / "candidates" / "candidates.csv", keep_default_na=False)
print(
    candidates.groupby("source_id").agg(
        n=("candidate_id", "size"),
        passed=("reasons", lambda r: (r == "").sum()),
        seconds=("seconds", "mean"),
        snr=("snr_db", "median"),
    )
)
print(candidates["reasons"].str.split(";").explode().value_counts())